In [1]:
import sys
sys.path.append('../../')

# Running a Refinement

In order to refine potential parameters the following is required:

- A `Universe` which contains one or more atoms. ([Building a Universe](building-a-universe.ipynb))
- A `Simulation` which runs using the `Universe`. ([Running a Simulation](running-a-simulation.ipynb))
- One or more `Parameters` to refine. ([Selecting fitting Parameters](selecting-fitting-parameters.ipynb))
- One or more experimental datasets.

For more information on how to produce the first three of these, please see the tutorials in the brackets.

## Experimental datasets

Whether or not an experimental dataset can be refined against depends on:

- if the relevant experimental `Observable` (e.g. dynamic structure factor, S(Q,w)) has been implemented.
- if a `Reader` for the specific file format of the data has been implemented (e.g. LAMP output file format).

If both of these exist, an experimental experimental dataset can be used.  If only the former has been implemented, then it would be necessary to add a subclass of the `ObservableReader` class to use the experimental dataset; more information about how to do is included within the `ObservableReader` documentation.

Each experimental dataset must be specified by a dictionary containing the following:

- file_name : A string with the file name of the experimental data
- type : A string specifying the type of `Observable` which describes the data (e.g. SQw if the data is the dynamic structure factor, PDF if the data is the pair distribution function).
- reader : A string specifying the name of the `ObservableReader` which will be used to read the file.
- weight : A float which determines the relative importance weighting of this dataset to other datasets.

In [2]:
QENS = {'file_name':'data/263K05Awat_LAMP',
        'type':'SQw',
        'reader':'LAMPSQw',
        'weight':1.}

The specific manner in which the `weight` applies to calculating the Figure of Merit (FoM) depends on the particular FoM which is used for the refinement. By default this an error normalized least squares difference (`StandardFoMCalculator`):

$FoM = \sum_{i} w_{i} \frac{F_{i}^{exp} - F_{i}^{sim}}{\sigma_{i}^{exp}}$

The `exp_datasets` parameter which is passed to `Control` is a list of all dataset dictionaries.  In this instance there is only a single experimental dataset:

In [18]:
exp_datasets = [QENS]

If another dataset were to be included, for example the pair distribution function, this would require its own dictionary:

In [16]:
n_diffraction = {'file_name':'data/water_PDF',
                 'type':'PDF',
                 'reader':'ASCII',
                 'weight':0.5}
two_exp_datasets = [QENS, n_diffraction]

## Controlling the refinement

Continuing from the [Running a Simulation](running-a-simulation.ipynb) tutorial, using a `Universe` filled with SPCE water molecules.  As this minimizes and equilibrates a small simulation, it may take ~90s to execute.

In [17]:
%%capture
# Run Running a Simulation notebook and hide output
%run "running-a-simulation.ipynb"

The `Control` object sets up and runs the refinement (using `Control.refine`).  It uses the experimental datasets to refine the fitting `Parameters` for the specified `Simulation`:

In [20]:
# Assuming a Universe called universe and a Simulation called simulation have been created
from MDMC.control.control import Control

universe = simulation.universe
fit_params = universe.parameters

control = Control(simulation=simulation,
                  exp_datasets=exp_datasets,
                  fit_params=fit_params,
                  MC_norm=1,
                  minimizer_type="MMC",
                  MD_steps=208000,
                  t_resolution=114.)

../../MDMC/trajectory_analysis/observables/SQw.py:490: NumbaPerformanceWarning: np.dot() is faster on contiguous arrays, called on (array(float64, 2d, A), array(float64, 1d, A))
  return np.exp(-1j * np.dot(Q_vector, r))


To refine the specified `fit_params`, pass the number of refinement steps to `Control.refine`.

If 0 steps are passed, an MD simulation is performed, the simulation `Observable` is calculated, and the FoM is calculated, however the `fit_params` are not modified.  This can be used to determine the agreement between a simulation with the current `fit_params` and the experimental data.

In [23]:
import numpy as np
from scipy.interpolate import interp2d
exp_obs = control.observable_pairs[0].exp_obs
Q_slice = slice(6, len(exp_obs.Q), 2)
Q = exp_obs.Q[Q_slice]
E_range = (exp_obs.E >=0)
E = exp_obs.E[E_range]
SQw = np.array([Sw[E_range] for Sw in exp_obs.SQw[Q_slice]])
SQw_err = np.array([Sw_err[E_range] for Sw_err
                    in exp_obs.SQw_err[Q_slice]])
SQw_fun = interp2d(E, Q, SQw)
SQw_err_zero = SQw_err
SQw_err_zero[SQw_err == np.float('inf')] = 0
SQw_err_fun = interp2d(E, Q, SQw_err_zero)
# Use the largest step size from the E data for the uniform step size
E_step = max([E[i] - E[i-1] for i in np.arange(len(E) - 1) + 1])
# Currently forced to start from E = 0. due to limitations of SQw calculation
E_uniform = np.arange(0., E[-1] - E[0], E_step )
SQw_uniform = SQw_fun(E_uniform, Q)
SQw_err_uniform = SQw_err_fun(E_uniform, Q)
SQw_err_uniform[SQw_err_uniform == 0.] = np.float('inf')
control.observable_pairs[0].exp_obs.independent_variables = {'E':E_uniform,
                                                             'Q':Q}
control.observable_pairs[0].exp_obs._dependent_variables = {'SQw':SQw_uniform}
control.observable_pairs[0].exp_obs._errors = {'SQw':SQw_err_uniform}
control.observable_pairs[0].MD_obs.independent_variables = {'E':E_uniform,
                                                            'Q':Q}

In [24]:
control.refine(0)

AssertionError: 
Not equal to tolerance rtol=1e-05, atol=0
Set E values and calculated E values are not consistent
Mismatch: 99%
Max absolute difference: 202.36806035
Max relative difference: 104.75643951
 x: array([  0.      ,   1.983494,   3.966988,   5.950481,   7.933975,
         9.917469,  11.900963,  13.884456,  15.86795 ,  17.851444,
        19.834938,  21.818431,  23.801925,  25.785419,  27.768913,...
 y: UnitNDArray([0.      , 0.018755, 0.037511, 0.056266, 0.075021, 0.093776,
             0.112532, 0.131287, 0.150042, 0.168798, 0.187553, 0.206308,
             0.225064, 0.243819, 0.262574, 0.28133 , 0.300085, 0.31884 ,...

## Observable Plotting

Plotting software can be used to visualize the experimental and simulation `Observable` objects.  Here `matplotlib` is used to define a function for plotting the dynamic structure factor, `SQw`.

In [ ]:
%matplotlib notebook  
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def plot_SQw(Q, E, SQw)
    fig = plt.figure()
    ax = fig.gca(projection='3d')
    X, Y = np.meshgrid(E, Q)
    surf = ax.plot_surface(Q, E, SQw)
    plt.show()

Each comparable experimental and simulation `Observable` will be part of an `ObservablePair`.  These will have the same order as the `exp_datasets` that were passed to `Control`.  So the QENS experimental data will be part of the first (zero index) `ObservablePair`:

In [ ]:
obs_pairs = control.observable_pairs
QENS_pair = obs_pairs[0]

Each `ObservablePair` must have an `Observable` from experimental data (`ObservablePair.exp_obs`) and an `Observable` from simulation ((`ObservablePair.MD_obs`).  In the case of the QENS `ObservablePair` (and other 2-D data), these can be plotted using the `plot_SQw` function defined above.

In [ ]:
E = QENS_pair.exp_obs.E
Q = QENS_pair.exp_obs.Q

# Plot the experimental SQw
plot_SQw(Q, E, QENS_pair.exp_obs.SQw)

In [ ]:
# Plot the simulation SQw
plot_SQw(Q, E, QENS_pair.MD_obs.SQw)